# Navier-Stokes Flow Around a Cylinder with Inlet Total Pressure

This notebook documents the Julia script that solves transient incompressible Navier-Stokes flow in a channel with a cylinder. The inlet is not prescribed by a velocity profile; instead, a total-pressure condition is imposed weakly at the inlet and explained in the math below.

## 1. Governing equations

The incompressible Navier-Stokes equations are

$$
\frac{\partial \mathbf{u}}{\partial t} + (\mathbf{u} \cdot \nabla)\mathbf{u} - \nu \nabla^2 \mathbf{u} + \nabla p = \mathbf{f},
$$

$$
\nabla \cdot \mathbf{u} = 0.
$$

Here $\mathbf{u}=(u_x,u_y)^T$ is the velocity, $p$ is pressure, $\nu$ is the kinematic viscosity, and $\mathbf{f}$ is the body force. In this script the forcing term is zero. The nonlinear term $(\mathbf{u}\cdot\nabla)\mathbf{u}$ makes the system nonlinear in the unknown velocity.

## 2. Weak form and finite elements

Testing with $\mathbf{v}$ and $q$, integrating over the domain $\Omega$, and applying the usual incompressible weak form gives

$$
\int_\Omega \frac{\partial \mathbf{u}}{\partial t} \cdot \mathbf{v}\, d\Omega
+ \int_\Omega [ (\mathbf{u} \cdot \nabla)\mathbf{u} ] \cdot \mathbf{v}\, d\Omega
+ \nu \int_\Omega \nabla \mathbf{u} : \nabla \mathbf{v}\, d\Omega
- \int_\Omega p \nabla \cdot \mathbf{v}\, d\Omega = 0,
$$

$$
\int_\Omega q \nabla \cdot \mathbf{u}\, d\Omega = 0.
$$

The code uses Taylor-Hood elements: quadratic velocity and linear pressure. This is the stable choice for incompressible flow.

In [ ]:
using Ferrite, SparseArrays, BlockArrays, LinearAlgebra, WriteVTK
using DiffEqBase
using OrdinaryDiffEqRosenbrock: Rodas5P

using FerriteGmsh
using FerriteGmsh: Gmsh

const nu = 1.0 / 1000.0
const rho = 2.0
const p0_inlet = 1.0
const alpha = 100.0
const t_ramp = 1.0
alpha_of_t(t) = alpha * min(t / t_ramp, 1.0)
const H_channel = 0.41
const tol_corner = 0.0011

## 3. Mesh and finite element spaces

The domain is a 2D channel with a circular cylinder removed from the center. The mesh is generated in Gmsh and tagged into bottom, left, right, top, and hole boundaries. Velocity uses $Q_2$ elements and pressure uses $Q_1$ elements on quadrilateral cells.

In [ ]:
dim = 2
Gmsh.initialize()
gmsh.option.set_number("General.Verbosity", 2)
rect_tag = gmsh.model.occ.add_rectangle(0, 0, 0, 1.1, 0.41)
circle_tag = gmsh.model.occ.add_circle(0.2, 0.2, 0, 0.05)
circle_curve_tag = gmsh.model.occ.add_curve_loop([circle_tag])
circle_surf_tag = gmsh.model.occ.add_plane_surface([circle_curve_tag])
gmsh.model.occ.cut([(dim, rect_tag)], [(dim, circle_surf_tag)])
gmsh.model.occ.synchronize()
gmsh.model.model.add_physical_group(dim - 1, [6], -1, "bottom")
gmsh.model.model.add_physical_group(dim - 1, [7], -1, "left")
gmsh.model.model.add_physical_group(dim - 1, [8], -1, "right")
gmsh.model.model.add_physical_group(dim - 1, [9], -1, "top")
gmsh.model.model.add_physical_group(dim - 1, [5], -1, "hole")
gmsh.option.setNumber("Mesh.Algorithm", 11)
gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 20)
gmsh.option.setNumber("Mesh.MeshSizeMax", 0.05)
gmsh.model.mesh.generate(dim)
grid = togrid()
Gmsh.finalize()

ip_v = Lagrange{RefQuadrilateral, 2}()^dim
ip_p = Lagrange{RefQuadrilateral, 1}()
ip_g = Lagrange{RefQuadrilateral, 1}()
qr = QuadratureRule{RefQuadrilateral}(4)
cellvalues_v = CellValues(qr, ip_v, ip_g)
cellvalues_p = CellValues(qr, ip_p, ip_g)
qr_facet = FacetQuadratureRule{RefQuadrilateral}(4)
facetvalues_v = FacetValues(qr_facet, ip_v, ip_g)
facetvalues_p = FacetValues(qr_facet, ip_p, ip_g)
dh = DofHandler(grid)
add!(dh, :v, ip_v)
add!(dh, :p, ip_p)
close!(dh)

## 4. Boundary conditions

The walls and cylinder use no-slip conditions, so $\mathbf{u}=\mathbf{0}$ on $\Gamma_{\text{walls}} \cup \Gamma_{\text{cyl}}$. The outlet is used only as a pressure gauge. The inlet is different: instead of prescribing velocity, the script enforces a total-pressure condition. For a horizontal inflow, the total pressure relation is

$$
p + \frac{1}{2}\rho u_x^2 = p_0,
$$

which is equivalent to the residual

$$
r_{tp}(u_x,p) = u_x^2 - \frac{2}{\rho}(p_0 - p) = 0.
$$

This condition is imposed weakly on the inlet boundary with a time-ramped penalty weight $\alpha(t)=\alpha\min(t/t_{\text{ramp}},1)$. Only the x-component is penalized because the inflow is horizontal. The corner points are skipped to avoid singular behavior where the inlet meets the no-slip walls.

In [ ]:
ch = ConstraintHandler(dh)
noslip_facet_names = ["top", "bottom", "hole"]
noslip_facets = union(getfacetset.((grid,), noslip_facet_names)...)
add!(ch, Dirichlet(:v, noslip_facets, (x, t) -> Vec((0.0, 0.0)), [1, 2]))
inlet_facets = getfacetset(grid, "left")
add!(ch, Dirichlet(:p, getfacetset(grid, "right"), (x, t) -> 0.0))
close!(ch)
update!(ch, 0.0)

@inline total_p_residual(ux, p_static) = ux^2 - 2.0 * (p0_inlet - p_static) / rho
@inline dtotal_p_residual_dux(ux) = 2.0 * ux
@inline dtotal_p_residual_dp() = 2.0 / rho

## 5. Mass matrix and Stokes operator

The mass matrix comes from the time derivative term,

$$
M_{ij} = \int_\Omega \phi_i \cdot \phi_j\, d\Omega,
$$

and only the velocity-velocity block is non-zero. The viscous operator contributes

$$
-\nu \int_\Omega \nabla \phi_i : \nabla \phi_j\, d\Omega,
$$

with the pressure coupling terms

$$
\int_\Omega (\nabla \cdot \phi_i)\psi_j\, d\Omega \qquad \text{and} \qquad \int_\Omega \psi_i (\nabla \cdot \phi_j)\, d\Omega.
$$

In [ ]:
function assemble_mass_matrix(cellvalues_v::CellValues, cellvalues_p::CellValues, M::SparseMatrixCSC, dh::DofHandler)
    n_v = getnbasefunctions(cellvalues_v)
    n_p = getnbasefunctions(cellvalues_p)
    M_e = BlockedArray(zeros(n_v + n_p, n_v + n_p), [n_v, n_p], [n_v, n_p])
    assembler = start_assemble(M)
    for cell in CellIterator(dh)
        fill!(M_e, 0)
        Ferrite.reinit!(cellvalues_v, cell)
        for qp in 1:getnquadpoints(cellvalues_v)
            dOmega = getdetJdV(cellvalues_v, qp)
            for i in 1:n_v
                phi_i = shape_value(cellvalues_v, qp, i)
                for j in 1:n_v
                    phi_j = shape_value(cellvalues_v, qp, j)
                    M_e[BlockIndex((1, 1), (i, j))] += phi_i ⋅ phi_j * dOmega
                end
            end
        end
        assemble!(assembler, celldofs(cell), M_e)
    end
    return M
end

function assemble_stokes_matrix(cellvalues_v::CellValues, cellvalues_p::CellValues, nu, K::SparseMatrixCSC, dh::DofHandler)
    n_v = getnbasefunctions(cellvalues_v)
    n_p = getnbasefunctions(cellvalues_p)
    K_e = BlockedArray(zeros(n_v + n_p, n_v + n_p), [n_v, n_p], [n_v, n_p])
    assembler = start_assemble(K)
    for cell in CellIterator(dh)
        fill!(K_e, 0)
        Ferrite.reinit!(cellvalues_v, cell)
        Ferrite.reinit!(cellvalues_p, cell)
        for qp in 1:getnquadpoints(cellvalues_v)
            dOmega = getdetJdV(cellvalues_v, qp)
            for i in 1:n_v
                grad_phi_i = shape_gradient(cellvalues_v, qp, i)
                for j in 1:n_v
                    grad_phi_j = shape_gradient(cellvalues_v, qp, j)
                    K_e[BlockIndex((1, 1), (i, j))] -= nu * grad_phi_i ⊡ grad_phi_j * dOmega
                end
            end
            for j in 1:n_p
                psi_j = shape_value(cellvalues_p, qp, j)
                for i in 1:n_v
                    div_phi_i = shape_divergence(cellvalues_v, qp, i)
                    K_e[BlockIndex((1, 2), (i, j))] += div_phi_i * psi_j * dOmega
                    K_e[BlockIndex((2, 1), (j, i))] += psi_j * div_phi_i * dOmega
                end
            end
        end
        assemble!(assembler, celldofs(cell), K_e)
    end
    return K
end

## 6. Inlet total-pressure residual

The inlet total-pressure condition is not imposed by fixing $u_x$ directly. Instead, the residual

$$
r_{tp}(u_x,p) = u_x^2 - \frac{2}{\rho}(p_0 - p),
$$

is integrated over the inlet boundary. In words: if the static pressure rises, the allowed inlet speed decreases, and if the static pressure drops, the permitted inlet speed increases. The penalty term is added only on the inlet facets away from the corners, so the singular corner interaction with the no-slip walls does not contaminate the condition.

In [ ]:
function add_inlet_total_pressure_rhs!(du, u, p::RHSparams, t)
    (; dh, facetvalues_v, facetvalues_p, inlet_facets) = p
    v_range = dof_range(dh, :v)
    p_range = dof_range(dh, :p)
    for facet in FacetIterator(dh, inlet_facets)
        Ferrite.reinit!(facetvalues_v, facet)
        Ferrite.reinit!(facetvalues_p, facet)
        celld = celldofs(facet)
        v_dofs = @view celld[v_range]
        p_dofs = @view celld[p_range]
        v_e = u[v_dofs]
        p_e = u[p_dofs]
        Re = zeros(length(v_dofs))
        for qp in 1:getnquadpoints(facetvalues_v)
            x = spatial_coordinate(facetvalues_v, qp, getcoordinates(facet))
            if x[2] <= tol_corner || x[2] >= H_channel - tol_corner
                continue
            end
            dGamma = getdetJdV(facetvalues_v, qp)
            ux = function_value(facetvalues_v, qp, v_e)[1]
            p_static = function_value(facetvalues_p, qp, p_e)
            res = total_p_residual(ux, p_static)
            for i in 1:getnbasefunctions(facetvalues_v)
                phi_x_i = shape_value(facetvalues_v, qp, i)[1]
                Re[i] -= alpha_of_t(t) * res * phi_x_i * dGamma
            end
        end
        assemble!(du, v_dofs, Re)
    end
    return
end

function add_inlet_total_pressure_jac!(J, u, p::RHSparams, t)
    (; dh, facetvalues_v, facetvalues_p, inlet_facets) = p
    v_range = dof_range(dh, :v)
    p_range = dof_range(dh, :p)
    assembler = start_assemble(J; fillzero = false)
    for facet in FacetIterator(dh, inlet_facets)
        Ferrite.reinit!(facetvalues_v, facet)
        Ferrite.reinit!(facetvalues_p, facet)
        celld = celldofs(facet)
        v_dofs = @view celld[v_range]
        p_dofs = @view celld[p_range]
        v_e = u[v_dofs]
        Jvv = zeros(length(v_dofs), length(v_dofs))
        for qp in 1:getnquadpoints(facetvalues_v)
            x = spatial_coordinate(facetvalues_v, qp, getcoordinates(facet))
            if x[2] <= tol_corner || x[2] >= H_channel - tol_corner
                continue
            end
            dGamma = getdetJdV(facetvalues_v, qp)
            ux = function_value(facetvalues_v, qp, v_e)[1]
            for i in 1:getnbasefunctions(facetvalues_v)
                phi_x_i = shape_value(facetvalues_v, qp, i)[1]
                for j in 1:getnbasefunctions(facetvalues_v)
                    phi_x_j = shape_value(facetvalues_v, qp, j)[1]
                    Jvv[i, j] -= alpha_of_t(t) * dtotal_p_residual_dux(ux) * phi_x_i * phi_x_j * dGamma
                end
                for j in 1:getnbasefunctions(facetvalues_p)
                    psi_j = shape_value(facetvalues_p, qp, j)
                    J[v_dofs[i], p_dofs[j]] -= alpha_of_t(t) * dtotal_p_residual_dp() * phi_x_i * psi_j * dGamma
                end
            end
        end
        assemble!(assembler, v_dofs, Jvv)
    end
    return
end

## 7. Residual, Jacobian, and time stepping

The full semidiscrete system is solved as a DAE. The residual combines the linear Stokes operator, the nonlinear convection term, and the inlet total-pressure contribution. The Jacobian is built from the same pieces, with the boundary term contributing the derivatives of $r_{tp}$ with respect to $u_x$ and $p$. The solver uses `Rodas5P` together with a limiter that re-applies the constraints at each step.

In [ ]:
struct RHSparams
    K::SparseMatrixCSC
    ch::ConstraintHandler
    dh::DofHandler
    cellvalues_v::CellValues
    facetvalues_v::FacetValues
    facetvalues_p::FacetValues
    inlet_facets
    u::Vector
end

T_end = 6.0
dt0 = 0.001
M = allocate_matrix(dh)
assemble_mass_matrix(cellvalues_v, cellvalues_p, M, dh)
K = allocate_matrix(dh)
assemble_stokes_matrix(cellvalues_v, cellvalues_p, nu, K, dh)
u0 = zeros(ndofs(dh))
apply!(u0, ch)
apply!(M, ch)
jac_sparsity = sparse(K)
params = RHSparams(K, ch, dh, cellvalues_v, facetvalues_v, facetvalues_p, inlet_facets, copy(u0))

function ferrite_limiter!(u, _, p, t)
    update!(p.ch, t)
    return apply!(u, p.ch)
end

function navierstokes!(du, u_uc, p::RHSparams, t)
    (; K, ch, dh, cellvalues_v, u) = p
    u .= u_uc
    update!(ch, t)
    apply!(u, ch)
    mul!(du, K, u)
    v_range = dof_range(dh, :v)
    n_basefuncs = getnbasefunctions(cellvalues_v)
    v_e = zeros(n_basefuncs)
    du_e = zeros(n_basefuncs)
    for cell in CellIterator(dh)
        Ferrite.reinit!(cellvalues_v, cell)
        v_celldofs = @view celldofs(cell)[v_range]
        v_e .= @views u[v_celldofs]
        fill!(du_e, 0.0)
        navierstokes_rhs_element!(du_e, v_e, cellvalues_v)
        assemble!(du, v_celldofs, du_e)
    end
    add_inlet_total_pressure_rhs!(du, u, p, t)
    return
end

function navierstokes_jac!(J, u_uc, p::RHSparams, t)
    (; K, ch, dh, cellvalues_v, u) = p
    u .= u_uc
    update!(ch, t)
    apply!(u, ch)
    nonzeros(J) .= nonzeros(K)
    assembler = start_assemble(J; fillzero = false)
    n_basefuncs = getnbasefunctions(cellvalues_v)
    J_e = zeros(n_basefuncs, n_basefuncs)
    v_e = zeros(n_basefuncs)
    v_range = dof_range(dh, :v)
    for cell in CellIterator(dh)
        Ferrite.reinit!(cellvalues_v, cell)
        v_celldofs = @view celldofs(cell)[v_range]
        v_e .= @views u[v_celldofs]
        fill!(J_e, 0.0)
        navierstokes_jac_element!(J_e, v_e, cellvalues_v)
        assemble!(assembler, v_celldofs, J_e)
    end
    add_inlet_total_pressure_jac!(J, u, p, t)
    return apply!(J, ch)
end

rhs = ODEFunction(navierstokes!, mass_matrix = M; jac = navierstokes_jac!, jac_prototype = jac_sparsity)
problem = ODEProblem(rhs, u0, (0.0, T_end), params)
timestepper = Rodas5P(autodiff = false, step_limiter! = ferrite_limiter!)
integrator = init(problem, timestepper; initializealg = NoInit(), dt = dt0, adaptive = true, abstol = 1.0e-4, reltol = 1.0e-5, progress = true, progress_steps = 1, verbose = true, internalnorm = FreeDofErrorNorm(ch), d_discontinuities = [t_ramp])

## 8. Output and runnable source

The solution is written to VTK for ParaView. The final code cell simply executes the original Julia source file so the notebook can be run end to end from this documentation copy.

In [ ]:
pvd = paraview_collection("sol/sol_ns_p0_inlet/navier-stokes-p0-inlet")
for (step, (u, t)) in enumerate(intervals(integrator))
    VTKGridFile("sol/sol_ns_p0_inlet/navier-stokes-p0-inlet-$(lpad(step, 4, '0'))", dh) do vtk
        write_solution(vtk, dh, u)
        pvd[t] = vtk
    end
end
vtk_save(pvd)